# Setup & Import

In [ ]:
import os
import warnings
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm
from PIL import Image
import numpy as np
import pandas as pd
import random
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# MAKE FOLDER

In [ ]:
os.makedirs("models", exist_ok=True)
os.makedirs("submission", exist_ok=True)
print("✅ Folder Ready")

# Advanced Transforms + Augmentation

In [ ]:
IMG_SIZE = 512   

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(35),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.12, 0.12), scale=(0.85, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 48),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Load Train Dataset 

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)   
val_loader   = DataLoader(val_dataset,   batch_size=24, shuffle=False, num_workers=4, pin_memory=True)

# Model ConvNeXt (Advanced)

In [ ]:
model_name = 'convnext_tiny'   

model = timm.create_model(
    model_name, 
    pretrained=True, 
    num_classes=3, 
    drop_path_rate=0.25,     
    img_size=512             
)

model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=8, T_mult=2)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Training Loop

In [ ]:
best_f1 = 0.0
num_epochs = 15

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
    
    # Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]", leave=False):
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    print(f"\nEpoch {epoch+1} - Loss: {train_loss/len(train_loader):.4f} | Val Macro F1: {val_f1:.4f}")
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "models/best_convnext.pth")
        print(f"✅ Best model saved! F1 = {best_f1:.4f}")
    
    scheduler.step()

# Test Prediction

In [ ]:
# Test Dataset 
class TestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = sorted([f for f in os.listdir(root_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.image_files[idx]

test_dataset = TestDataset("data/test", transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0, pin_memory=False)  # batch kecil

In [ ]:
# Prediction
model.load_state_dict(torch.load("models/best_convnext.pth", weights_only=True))
model.eval()

submission = []

with torch.no_grad():
    for images, _ in tqdm(test_loader, desc="Predicting"):
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        for p in preds:
            submission.append({'id': len(submission), 'predicted': int(p)})

sub_df = pd.DataFrame(submission)
sub_df.to_csv('submission_NamaTim.csv', index=False)
print("✅ Submission selesai! Total:", len(sub_df))
print(sub_df.head())